# Lesson 10 - AI Agents in Production

In this lesson you will learn **production patterns** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **Observability** — adding timing and logging to agent interactions
- **Evaluation** — using an evaluator agent to score response quality
- **Cost management** — strategies for token optimization and model selection

The scenario is a **travel agent** that helps users plan trips, with monitoring and evaluation layered on top.


## ਸੈੱਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਪ੍ਰੋਡਕਸ਼ਨ ਵਿਚ ਵਿਚਾਰ

AI ਏਜੰਟਾਂ ਨੂੰ ਪ੍ਰੋਟੋਟਾਈਪ ਤੋਂ ਪ੍ਰੋਡਕਸ਼ਨ ਵਿਚ ਲਿਜਾਣ ਲਈ ਤਿੰਨ ਸਤੰਭਾਂ 'ਤੇ ਧਿਆਨ ਦੇਣਾ ਜ਼ਰੂਰੀ ਹੈ:

1. **ਦਿੱਖਤਾ** — ਤੁਹਾਨੂੰ ਇਹ ਵੇਖਣਾ ਲੋੜੀਂਦਾ ਹੈ ਕਿ ਏਜੰਟ ਕੀ ਕਰ ਰਿਹਾ ਹੈ, ਕਿੰਨਾ ਸਮਾਂ ਲੱਗਦਾ ਹੈ, ਅਤੇ ਕਿਹੜੇ ਟੂਲ ਕਾਲ ਕਰਦਾ ਹੈ। ਟਰੇਸਿੰਗ ਅਤੇ ਲੋਗਿੰਗ ਬਿਨਾਂ, ਪ੍ਰੋਡਕਸ਼ਨ ਸਮੱਸਿਆਵਾਂ ਨੂੰ ਡੀਬੱਗ ਕਰਨਾ ਲਗਭਗ ਅਸੰਭਵ ਹੈ।

2. **ਮੁਲਾਂਕਣ** — ਸਵੈਚਾਲਿਤ ਗੁਣਵੱਤਾ ਚੈੱਕ ਇਹ ਯਕੀਨੀ ਬਣਾਉਂਦੇ ਹਨ ਕਿ ਏਜੰਟ ਦੇ ਜਵਾਬ ਸਮੇਂ ਦੇ ਨਾਲ ਸਹੀ, ਪੂਰੇ, ਅਤੇ ਮਦਦਗਾਰ ਰਹਿੰਦੇ ਹਨ। ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੰਟ ਪੱਧਰ-ਬੱਧ ਮਾਪਦੰਡਾਂ ਦੇ ਮੁਤਾਬਕ ਜਵਾਬਾਂ ਨੂੰ ਗ੍ਰੇਡ ਕਰ ਸਕਦਾ ਹੈ।

3. **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਵਰਤੋਂ ਸਿੱਧੀ ਤਰ੍ਹਾਂ ਲਾਗਤ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰਦੀ ਹੈ। ਪ੍ਰੰਪਟ ਅਪਟੀਮਾਈਜੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ, ਅਤੇ ਕੈਸ਼ਿੰਗ ਵਰਗੀਆਂ ਰਣਨੀਤੀਆਂ ਮਿਆਰੀਤਾ ਨੂੰ ਬਿਨਾਂ ਘਟਾਏ ਖਰਚੇ ਕਾਬੂ ਵਿੱਚ ਰੱਖਣ ਵਿੱਚ ਸਹਾਇਤਾ ਕਰਦੀਆਂ ਹਨ।


## ਇੱਕ ਦਰਸ਼ਨੀ ਏਜੰਟ ਬਣਾਉਣਾ

ਅਸੀਂ ਯਾਤਰਾ ਦੇ ਸੰਦ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਏਜੰਟ ਕਾਲ ਨੂੰ ਸਮੇਂ ਨਾਲ ਲਪੇਟਦੇ ਹਾਂ ਤਾਂ ਜੋ ਅਸੀਂ ਦੇਰੀ ਨੂੰ ਨਿਗਰਾਨੀ ਕਰ ਸਕੀਏ। ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ OpenTelemetry ਜਾਂ ਇਸੇ ਤਰ੍ਹਾਂ ਦੇ ਟਰੇਸਿੰਗ ਬੈਕਐਂਡ ਨਾਲ ਇੰਟੀਗਰੇਟ ਕਰੋਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ਮੁਲਾਂਕਣ ਪੈਟਰਨ

ਇੱਕ ਆਮ ਉਤਪਾਦਨ ਪੈਟਰਨ ਦੂਸਰੇ ਏਜੰਟ ਨੂੰ **ਮੁਲਾਂਕਣਕਾਰ** ਵਜੋਂ ਵਰਤਣ ਦਾ ਹੈ। ਮੁਲਾਂਕਣਕਾਰ ਪ੍ਰਾਇਮਰੀ ਏਜੰਟ ਦੇ ਜਵਾਬ ਨੂੰ ਪੂਰਣਤਾ, ਸਹੀਪਣ ਅਤੇ ਮਦਦਗਾਰਤਾ ਵਰਗੇ ਪੂਰਵ ਨਿਰਧਾਰਿਤ ਮਾਪਦੰਡਾਂ ਦੇ ਅਧਾਰ 'ਤੇ ਸਕੋਰ ਕਰਦਾ ਹੈ।

ਇਸ ਨਾਲ ਇਹ ਸੰਭਵ ਹੁੰਦਾ ਹੈ:
- ਉਪਭੋਗਤਾਵਾਂ ਤੱਕ ਜਵਾਬ ਪਹੁੰਚਣ ਤੋਂ ਪਹਿਲਾਂ ਸੁਚੱਜੇ ਗੁਣਵੱਤਾ ਗੇਟਸ ਆਟੋਮੇਟਿਕ ਖ਼ਾਤਰ
- ਜਦੋਂ ਪ੍ਰੰਪਟ ਜਾਂ ਮਾਡਲ ਬਦਲਦੇ ਹਨ ਤਾਂ ਰਿਗ੍ਰੈਸ਼ਨ ਪਤਾ ਲਗਾਉਣਾ
- ਸਮੇਂ ਦੇ ਨਾਲ ਏਜੰਟ ਦੀ ਕਾਰਗੁਜ਼ਾਰੀ ਦੀ ਲਗਾਤਾਰ ਨਿਗਰਾਨੀ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ਲਾਗਤ ਪ੍ਰਬੰਧਨ ਦੀਆਂ ਰਣਨੀਤੀਆਂ

ਉਤਪਾਦਨ ਏਆਈ ਏਜੰਟਸ ਲਈ ਲਾਗਤਾਂ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਨਾ ਬਹੁਤ ਜਰੂਰੀ ਹੈ। ਇੱਥੇ ਮੁੱਖ ਰਣਨੀਤੀਆਂ ਹਨ:

| ਰਣਨੀਤੀ | ਵਰਣਨ |
|---|---|
| **ਪ੍ਰੰਪਟ ਅਪਟਮਾਈਜੇਸ਼ਨ** | ਸਿਸਟਮ ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸੰਖੇਪ ਰੱਖੋ। ਇਨਪੁੱਟ ਟੋਕਨਾਂ ਨੂੰ ਘਟਾਉਣ ਲਈ ਅਣਵਾਜ਼ੀ ਸੰਦਰਭ ਹਟਾਓ। |
| **ਮਾਡਲ ਚੋਣ** | ਸਧਾਰਣ ਕਾਰਜਾਂ ਜਿਵੇਂ ਕਿ ਵਰਗੀਕਰਨ ਜਾਂ ਨਿਕਾਸ ਲਈ ਛੋਟੇ, ਸਸਤੇ ਮਾਡਲ (ਜਿਵੇਂ GPT-4o-mini) ਵਰਤੋ, ਅਤੇ ਜਟਿਲ ਵਿਚਾਰਧਾਰਾ ਲਈ ਵੱਡੇ ਮਾਡਲ ਰੱਖੋ। |
| **ਕੈਸ਼ਿੰਗ** | ਦੁਹਰਾਉਣ ਵਾਲੀਆਂ ਟੂਲ ਨਤੀਜੇ ਅਤੇ ਅਕਸਰ ਪੁੱਛੀਆਂ ਜਾਣ ਵਾਲੀਆਂ ਪੜਤਾਲਾਂ ਨੂੰ ਕੈਸ਼ ਕਰੋ ਤਾਂ ਜੋ ਫਿਰ ਤੋਂ ਏਪੀਆਈ ਕਾਲ ਕਰਨ ਤੋਂ ਬਚਿਆ ਜਾ ਸਕੇ। |
| **ਟੋਕਨ ਬਜਟ** | ਅਣਪਛਾਤੇ ਲੰਬੇ ਜਵਾਬਾਂ ਨੂੰ ਰੋਕਣ ਲਈ `max_tokens` ਸੀਮਾਵਾਂ ਸੈੱਟ ਕਰੋ। |
| **ਬੈਚਿੰਗ** | ਜਿੱਥੇ ਸੰਭਵ ਹੋਵੇ, ਬਹੁਤ ਸਾਰੇ ਯੂਜ਼ਰ ਪੁੱਛਗਿੱਛਾਂ ਨੂੰ ਇੱਕੋ ਏਪੀਆਈ ਕਾਲ ਵਿੱਚ ਗਰੁੱਪ ਕਰੋ। |

ਵਾਸਤਵ ਵਿੱਚ, ਇੱਕ ਸਤਰ-ਬੱਧ ਪਹੁੰਚ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕੰਮ ਕਰਦੀ ਹੈ: ਸਿੱਧੇ ਸਧਾਰਨ ਵਿੰਨਤੀ ਨੂੰ ਤੇਜ਼, ਸਸਤੇ ਮਾਡਲ ਵੱਲ ਰੁਟ ਕਰੋ ਅਤੇ ਸਿਰਫ ਕਠਿਨ ਪੁੱਛਗਿੱਛਾਂ ਨੂੰ ਇੱਕ ਹੋਰ ਸਮਰੱਥ ਮਾਡਲ ਵੱਲ ਵਧਾਓ।


## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ:

1. ਏਜੰਟ ਦੇ ਇੰਟਰਐਕਸ਼ਨਾਂ ਵਿੱਚ ਸਮਾਂ ਅਤੇ ਲਾਗਿੰਗ ਦੇ ਨਾਲ **ਨਿਰੀਖਣਯੋਗਤਾ ਜੋੜੀ ਜਾਵੇ**, ਜੋ ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਮਾਨੀਟਰਿੰਗ ਲਈ ਮੰਜ਼ਿਲ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. ਇੱਕ ਮೌಲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਜਵਾਬਾਂ ਨੂੰ ਖੁਦਕੁਸ਼ੀ ਤੌਰ 'ਤੇ **ਮੁਲਾਂਕਣ ਕਰਨਾ**, ਜੋ ਪੂਰੀਤਾ, ਸਹੀਤਾ ਅਤੇ ਮਦਦਗਾਰਤਾ ਨੂੰ ਅੰਕ ਦਿੰਦਾ ਹੈ।
3. ਪ੍ਰਾਂਪਟ ਅਨੁਕੂਲਨ, ਮਾਡਲ ਚੋਣ, ਕੈਸ਼ਿੰਗ ਅਤੇ ਟੋਕਨ ਬਜਟਾਂ ਰਾਹੀਂ **ਲਾਗਤਾਂ ਦਾ ਪ੍ਰਬੰਧਨ ਕਰਨਾ**।

ਇਹ ਉਤਪਾਦਨ ਪੈਟਰਨ ਯਕੀਨੀ ਬਣਾਉਂਦੇ ਹਨ ਕਿ ਤੁਹਾਡੇ ਏਆਈ ਏਜੰਟ ਭਰੋਸੇਮੰਦ, ਮਾਪਣਯੋਗ ਅਤੇ ਲਾਗਤ ਪ੍ਰਭਾਵਸ਼ালী ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
